# Exact validation of five candidate inequalities for the siglocal cycle polytope

This notebook performs one narrow task, independently of the old LP and NPA code:

1. generate the corrected deterministic vertices for the clockwise and counterclockwise cycle orientations;
2. form their distinct union, which is the vertex set used for the full siglocal polytope;
3. encode five candidate inequalities exactly; and
4. compute the exact minimum of every functional on each orientation and on their union.

No optimization solver is used. Every coefficient is an exact integer or rational number, and every classical value is computed exactly. The only nonexact calculation is the optional affine-dimension diagnostic.


## Coordinate convention

Every behaviour is a 64-component vector containing $p(abc|xyz)$. The input blocks are ordered as

```text
(x,y,z) = 000, 001, 010, 011, 100, 101, 110, 111.
```

Inside each block, the outputs are ordered as

```text
(a,b,c) = 000, 001, 010, 011, 100, 101, 110, 111.
```

Thus $c$ and $z$ are the fastest-changing indices. In every displayed $8\times8$ coefficient matrix, rows correspond to $(x,y,z)$ and columns correspond to $(a,b,c)$. The conversion functions below make this convention explicit instead of relying on Julia's implicit matrix flattening order.


In [3]:
using LinearAlgebra

# These explicit lists avoid any ambiguity about which binary index changes fastest.
CONTEXTS = [
    (0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1),
    (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1),
]
OUTPUTS = copy(CONTEXTS)

# Position of p(a,b,c|x,y,z) in the canonical 64-vector.
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

# Row and column positions in an 8 x 8 coefficient matrix.
context_index(x, y, z) = 1 + z + 2*y + 4*x
outcome_index(a, b, c) = 1 + c + 2*b + 4*a

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in OUTPUTS] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in CONTEXTS] == collect(1:8:57)

println("Canonical coordinate convention initialized.")


Canonical coordinate convention initialized.


## Corrected deterministic siglocal vertices

A deterministic binary response depending on two binary inputs is one of $2^4=16$ functions. We encode its four outputs by the four bits of an integer from 0 to 15.

For the clockwise orientation we use

$$a=f_A(x,z),\qquad b=f_B(y,x),\qquad c=f_C(z,y).$$

For the counterclockwise orientation we use

$$a=f_A(x,y),\qquad b=f_B(y,z),\qquad c=f_C(z,x).$$

There are $16^3=4096$ deterministic strategies in each orientation. Their intersection contains 64 local deterministic behaviours, so the distinct union must contain $4096+4096-64=8128$ vertices.


In [5]:
"""Evaluate a deterministic binary function of two binary inputs."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one corrected deterministic behaviour in canonical coordinates."""
function deterministic_vertex(fA::Int, fB::Int, fC::Int, orientation::Symbol)
    vertex = zeros(Int8, 64)

    for (x, y, z) in CONTEXTS
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate all 4096 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    @assert row == 16^3 + 1
    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

# This diagnostic uses floating-point linear algebra only to report dimensions.
# It is not used to determine any inequality bound.
function affine_dimension(points; atol=1e-9)
    number_of_points = size(points, 1)
    number_of_points == 0 && return -1
    number_of_points == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)
vertices_siglocal = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)
intersection_size = length(intersect(clockwise_set, counterclockwise_set))

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_siglocal)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert intersection_size == 64
@assert size(vertices_siglocal, 1) == 8128
@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_siglocal) == 50

println("Corrected siglocal vertex set")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", intersection_size)
println("  distinct union:            ", size(vertices_siglocal, 1))
println("  affine dimension:          ", affine_dimension(vertices_siglocal))


Corrected siglocal vertex set
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## Exact encoding of the five inequalities

The first three matrices reproduce the displayed rounded coefficient matrices associated with `M_29`, `M29_disturbed1`, and `M29_disturbed2` in the old notebook. Their entries are treated as exact elements of $\{0,1/2,1\}$.

`Ineq_clock` and `Ineq_cycle` are encoded with their displayed integer coefficients. The proposed inequalities are all lower-bound inequalities:

- the three class-29 candidates have proposed bound 1;
- `Ineq_clock` has proposed bound 2;
- `Ineq_cycle` has proposed bound 20.

Matrix variables and vector variables have different names throughout this notebook.


In [7]:
"""Convert an 8 x 8 coefficient matrix to the canonical 64-vector."""
function matrix_to_canonical_vector(matrix)
    @assert size(matrix) == (8, 8)
    coefficients = Vector{eltype(matrix)}(undef, 64)

    for (x, y, z) in CONTEXTS, (a, b, c) in OUTPUTS
        coefficients[prob_index(a, b, c, x, y, z)] =
            matrix[context_index(x, y, z), outcome_index(a, b, c)]
    end

    return coefficients
end

"""Convert a canonical 64-vector back to its 8 x 8 matrix display."""
function canonical_vector_to_matrix(coefficients)
    @assert length(coefficients) == 64
    matrix = Matrix{eltype(coefficients)}(undef, 8, 8)

    for (x, y, z) in CONTEXTS, (a, b, c) in OUTPUTS
        matrix[context_index(x, y, z), outcome_index(a, b, c)] =
            coefficients[prob_index(a, b, c, x, y, z)]
    end

    return matrix
end


canonical_vector_to_matrix

In [8]:
# Original class-29 LP inequality: M_29, equivalently c29_opt_round.
M_29_matrix = Rational{Int}[
    0    1    1  1//2    1  1//2  1//2    0;
 1//2    0  1//2  1//2    0     1     0    0;
    0    0    0     1  1//2     0  1//2    0;
    1    1    1     0    0  1//2     0  1//2;
    0  1//2    1     0    0  1//2     1    0;
 1//2    1    0     0    1     0  1//2  1//2;
    1    0    0  1//2    1     0     0  1//2;
    1  1//2  1//2    0  1//2     0     0    1
]
c29_opt_round = matrix_to_canonical_vector(M_29_matrix)

# First disturbed class-29 inequality.
M29_disturbed1_matrix = Rational{Int}[
    0    1    1  1//2    1  1//2  1//2    0;
    0    0  1//2  1//2    1     1     0    0;
    0    1    0     1  1//2     0  1//2    0;
    1    0    1     0    0  1//2     0  1//2;
    0  1//2    1     0    0  1//2     1    0;
    1    1    0     0    0     0  1//2  1//2;
    1    0    0  1//2    1     0     0  1//2;
    1  1//2  1//2    0  1//2     0     0    1
]
c29_disturbed_round1 = matrix_to_canonical_vector(M29_disturbed1_matrix)

# Second disturbed class-29 inequality.
M29_disturbed2_matrix = Rational{Int}[
    0    1    1  1//2    1  1//2  1//2    0;
 1//2    0    1  1//2    0     1  1//2    0;
 1//2    0    0     1    1  1//2  1//2    0;
 1//2    1    1     0    1     0     0  1//2;
 1//2    1    0  1//2    0  1//2     1    0;
 1//2    1    1     0    1     0     0  1//2;
 1//2    1    1     0    1     0     0  1//2;
    1  1//2  1//2    0  1//2     0     0    1
]
c29_disturbed_round2 = matrix_to_canonical_vector(M29_disturbed2_matrix)

# Clockwise-only inequality.
Ineq_clock_matrix = Rational{Int}[
    0  1  1  2  1  2  2  0;
    0  0  2  2  1  1  0  0;
    0  1  0  1  2  0  2  0;
    1  0  1  0  0  2  0  2;
    0  2  1  0  0  2  1  0;
    1  1  0  0  0  0  2  2;
    1  0  0  2  1  0  0  2;
    2  0  0  1  0  1  1  2
]
Ineq_clock_vec = matrix_to_canonical_vector(Ineq_clock_matrix)

# Full-cycle inequality.
Ineq_cycle_matrix = Rational{Int}[
    0  12  12  15  12  15  15   0;
    0   4  15  12  15  12   5   0;
    0  15   4  12  15   5  12   0;
    8   4   4   0   0   5   5  10;
    0  15  15   5   4  12  12   0;
    8   4   0   5   4   0   5  10;
    8   0   4   5   4   5   0  10;
    5   0   0   4   0   4   4   8
]
Ineq_cycle_vec = matrix_to_canonical_vector(Ineq_cycle_matrix)

# Round-trip assertions protect against row/column transpositions.
@assert canonical_vector_to_matrix(c29_opt_round) == M_29_matrix
@assert canonical_vector_to_matrix(c29_disturbed_round1) == M29_disturbed1_matrix
@assert canonical_vector_to_matrix(c29_disturbed_round2) == M29_disturbed2_matrix
@assert canonical_vector_to_matrix(Ineq_clock_vec) == Ineq_clock_matrix
@assert canonical_vector_to_matrix(Ineq_cycle_vec) == Ineq_cycle_matrix

candidates = [
    (name="M_29 / c29_opt_round", coefficients=c29_opt_round, proposed_bound=1//1),
    (name="M29_disturbed1 / c29_disturbed_round1", coefficients=c29_disturbed_round1, proposed_bound=1//1),
    (name="M29_disturbed2 / c29_disturbed_round2", coefficients=c29_disturbed_round2, proposed_bound=1//1),
    (name="Ineq_clock", coefficients=Ineq_clock_vec, proposed_bound=2//1),
    (name="Ineq_cycle", coefficients=Ineq_cycle_vec, proposed_bound=20//1),
]

@assert length(Set(Tuple(candidate.coefficients) for candidate in candidates)) == 5
println("Encoded five distinct coefficient vectors exactly.")


Encoded five distinct coefficient vectors exactly.


## Exact audit on every deterministic vertex

Because the siglocal set is the convex hull of the deterministic vertices from the two orientations, the minimum of a linear functional over the entire polytope is the minimum attained on this finite vertex set.

A proposed lower-bound inequality $I(p)\geq\beta$ is valid for the full siglocal polytope if and only if the exact minimum over the distinct 8128 vertices is at least $\beta$.


In [10]:
"""Evaluate one rational functional exactly on every supplied vertex."""
function exact_vertex_values(vertices, coefficients::Vector{Rational{Int}})
    @assert size(vertices, 2) == length(coefficients)
    values = Vector{Rational{Int}}(undef, size(vertices, 1))

    for row in axes(vertices, 1)
        total = 0//1
        @inbounds for column in axes(vertices, 2)
            # Every deterministic vertex entry is exactly zero or one.
            if vertices[row, column] == 1
                total += coefficients[column]
            end
        end
        values[row] = total
    end

    return values
end

function audit_candidate(candidate)
    clockwise_values = exact_vertex_values(vertices_clockwise, candidate.coefficients)
    counterclockwise_values = exact_vertex_values(vertices_counterclockwise, candidate.coefficients)
    siglocal_values = exact_vertex_values(vertices_siglocal, candidate.coefficients)

    clockwise_minimum = minimum(clockwise_values)
    counterclockwise_minimum = minimum(counterclockwise_values)
    siglocal_minimum = minimum(siglocal_values)

    # The union minimum must equal the smaller orientation minimum.
    @assert siglocal_minimum == min(clockwise_minimum, counterclockwise_minimum)

    return (
        name=candidate.name,
        proposed_bound=candidate.proposed_bound,
        clockwise_minimum=clockwise_minimum,
        counterclockwise_minimum=counterclockwise_minimum,
        siglocal_minimum=siglocal_minimum,
        clockwise_saturation_count=count(==(clockwise_minimum), clockwise_values),
        counterclockwise_saturation_count=count(==(counterclockwise_minimum), counterclockwise_values),
        siglocal_saturation_count=count(==(siglocal_minimum), siglocal_values),
        valid_full_siglocal=(siglocal_minimum >= candidate.proposed_bound),
    )
end

audit_results = audit_candidate.(candidates)

# These assertions make the notebook stop loudly if any result differs from
# the independently checked values expected for the five stored matrices.
expected_results = Dict(
    # Each tuple stores: clockwise minimum, counterclockwise minimum,
    # union minimum, and the corresponding three saturation counts,
    # followed by the full-siglocal validity flag.
    "M_29 / c29_opt_round" => (1//1, 0//1, 0//1, 216, 8, 8, false),
    "M29_disturbed1 / c29_disturbed_round1" => (1//1, 0//1, 0//1, 352, 6, 6, false),
    "M29_disturbed2 / c29_disturbed_round2" => (1//1, 0//1, 0//1, 129, 3, 3, false),
    "Ineq_clock" => (2//1, 0//1, 0//1, 352, 6, 6, false),
    "Ineq_cycle" => (20//1, 20//1, 20//1, 136, 136, 261, true),
)

for result in audit_results
    expected = expected_results[result.name]
    obtained = (
        result.clockwise_minimum,
        result.counterclockwise_minimum,
        result.siglocal_minimum,
        result.clockwise_saturation_count,
        result.counterclockwise_saturation_count,
        result.siglocal_saturation_count,
        result.valid_full_siglocal,
    )
    @assert obtained == expected "Unexpected audit result for $(result.name): $obtained"
end

rational_string(value::Rational) =
    denominator(value) == 1 ? string(numerator(value)) : string(value)

println("Exact validation results")
println(join([
    "candidate",
    "proposed bound",
    "clockwise minimum",
    "counterclockwise minimum",
    "full siglocal minimum",
    "valid for full siglocal?",
], '\t'))

for result in audit_results
    println(join([
        result.name,
        rational_string(result.proposed_bound),
        rational_string(result.clockwise_minimum),
        rational_string(result.counterclockwise_minimum),
        rational_string(result.siglocal_minimum),
        result.valid_full_siglocal ? "YES" : "NO",
    ], '\t'))
end

println("\nAll exact-result assertions passed.")


Exact validation results
candidate	proposed bound	clockwise minimum	counterclockwise minimum	full siglocal minimum	valid for full siglocal?
M_29 / c29_opt_round	1	1	0	0	NO
M29_disturbed1 / c29_disturbed_round1	1	1	0	0	NO
M29_disturbed2 / c29_disturbed_round2	1	1	0	0	NO
Ineq_clock	2	2	0	0	NO
Ineq_cycle	20	20	20	20	YES

All exact-result assertions passed.


In [11]:
# Save the same audit table in a plain TSV file for later inspection.
OUTPUT_PATH = joinpath(pwd(), "cycle_five_inequality_validation.tsv")

open(OUTPUT_PATH, "w") do io
    println(io, join([
        "candidate",
        "proposed_bound",
        "clockwise_minimum",
        "counterclockwise_minimum",
        "full_siglocal_minimum",
        "clockwise_saturation_count",
        "counterclockwise_saturation_count",
        "full_siglocal_saturation_count",
        "valid_full_siglocal",
    ], '\t'))

    for result in audit_results
        println(io, join([
            result.name,
            rational_string(result.proposed_bound),
            rational_string(result.clockwise_minimum),
            rational_string(result.counterclockwise_minimum),
            rational_string(result.siglocal_minimum),
            string(result.clockwise_saturation_count),
            string(result.counterclockwise_saturation_count),
            string(result.siglocal_saturation_count),
            string(result.valid_full_siglocal),
        ], '\t'))
    end
end

println("Saved exact audit table to:")
println(abspath(OUTPUT_PATH))


Saved exact audit table to:
C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_five_inequality_validation.tsv


## Interpretation

The first three class-29 coefficient vectors and `Ineq_clock` attain value zero on counterclockwise deterministic vertices. Consequently, their proposed positive lower bounds do not define valid inequalities of the full siglocal polytope.

`Ineq_cycle` has exact minimum 20 in both orientations and therefore satisfies

$$I_{\mathrm{cycle}}(p)\geq20$$

for every behaviour in the full siglocal polytope. This notebook intentionally stops at the classical exact audit. Quantum/NPA analysis should be performed in a separate next-stage notebook, using only candidates that pass this validation.
